# 🌐 Global AI Layoffs — Data Cleaning & Preprocessing
**Dataset:** [Kaggle – Global AI Layoffs and Job Market 2020–Present](https://www.kaggle.com/datasets/belbino/global-ai-layoffs-and-job-market-2020-present)


---

## 📌 Business Goals

| # | Goal | KPIs |
|---|------|------|
| 1 | Which industries were most affected? | Total layoffs by industry, avg layoffs per company |
| 2 | How did layoffs trend over time? | Quarterly trends, year-over-year changes |
| 3 | Are AI companies more resilient? | Avg % workforce laid off (AI vs non-AI) |
| 4 | Which cities/countries had the highest reductions? | Total layoffs by country, top 10 cities |
| 5 | Which funding stages are most vulnerable? | Avg layoffs by stage, layoff event frequency |

---

## 🔧 Cleaning Steps
1. Import libraries & load data  
2. Initial inspection  
3. Fix data types  
4. Handle missing values  
5. Standardize text fields  
6. Feature engineering  
7. Final validation & export

In [50]:
import pandas as pd
import numpy as np
import os
import zipfile
from dotenv import load_dotenv
from kaggle.api.kaggle_api_extended import KaggleApi

## 1. Load Data

In [51]:
# Load credentials from .env file
load_dotenv()

os.environ['KAGGLE_USERNAME'] = os.getenv('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = os.getenv('KAGGLE_KEY')

def fetch_automated_layoffs_df():
    api = KaggleApi()
    api.authenticate()

    dataset_slug = "belbino/global-ai-layoffs-and-job-market-2020-present"
    target_folder = "../data/"
    target_file = "layoffs_events.csv"

    # Create the folder if it doesn't exist yet
    os.makedirs(target_folder, exist_ok=True)

    print("Connecting to Kaggle using 'Layoff_Project' token...")
    api.dataset_download_files(dataset_slug, path=target_folder, unzip=False)

    zip_path = os.path.join(target_folder, "global-ai-layoffs-and-job-market-2020-present.zip")
    extracted_file_path = os.path.join(target_folder, target_file)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        if target_file in zip_ref.namelist():
            zip_ref.extract(target_file, target_folder)
            print(f"Successfully extracted: {target_file}")
        else:
            print(f"Error: {target_file} not found in zip.")
            import sys
            sys.exit(1) # Clean exit using sys instead of the bare exit()

    if os.path.exists(zip_path):
        os.remove(zip_path)

    df = pd.read_csv(extracted_file_path)
    return df

df = fetch_automated_layoffs_df()

Connecting to Kaggle using 'Layoff_Project' token...
Dataset URL: https://www.kaggle.com/datasets/belbino/global-ai-layoffs-and-job-market-2020-present
Successfully extracted: layoffs_events.csv


In [4]:
print(f"Shape: {df.shape}")
df.head()

Shape: (2470, 11)


,company,location,layoff_count,date,pct_workforce,industry,source_url,stage,raised_mm,country,is_ai_company
0,Panda Squad,SF Bay Area,6.0,2020-03-13,75%,Consumer,https://twitter.com/danielsing er/status/12385...,Seed,$1.00,United States,False
1,HopSkipDrive,Los Angeles,8.0,2020-03-13,10%,Transportat…,https://layoffs.fyi/2020/04/02/ hopskipdrive-l...,Unknown,$45.00,United States,False
2,Help.com,Austin,16.0,2020-03-16,100%,Support,LinkedIn,Seed,$6.00,United States,False
3,Service,Los Angeles,NaN,2020-03-16,100%,Travel,https://techcrunch.com/2020/ 03/16/travel-savi...,Seed,$5.00,United States,False
4,Inspirato,Denver,130.0,2020-03-16,22%,Travel,https://businessden.com/2020 /03/16/inspirato-...,Series C,$79.00,United States,False


## 2. Initial Inspection

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2470 entries, 0 to 2469
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   company        2470 non-null   str    
 1   location       2470 non-null   str    
 2   layoff_count   1592 non-null   float64
 3   date           2470 non-null   str    
 4   pct_workforce  1534 non-null   str    
 5   industry       2468 non-null   str    
 6   source_url     2469 non-null   str    
 7   stage          2465 non-null   str    
 8   raised_mm      2179 non-null   str    
 9   country        2468 non-null   str    
 10  is_ai_company  2470 non-null   bool   
dtypes: bool(1), float64(1), str(9)
memory usage: 195.5 KB


In [6]:
df.describe()

,layoff_count
count,1592.000000
mean,340.008166
std,1397.407383
min,4.000000
25%,40.000000
50%,89.500000
75%,200.000000
max,30000.000000


In [7]:
df.dtypes

company              str
location             str
layoff_count     float64
date                 str
pct_workforce        str
industry             str
source_url           str
stage                str
raised_mm            str
country              str
is_ai_company       bool
dtype: object

In [16]:
missing = df.isnull().sum().reset_index()
missing.columns = ["Column", "Missing Count"]
missing["Missing %"] = (missing["Missing Count"] / len(df) * 100).round(2)
missing[missing["Missing Count"] > 0].sort_values("Missing %", ascending=False)

,Column,Missing Count,Missing %
4,pct_workforce,936,37.89
2,layoff_count,878,35.55
8,raised_mm,291,11.78
7,stage,5,0.20
5,industry,2,0.08
9,country,2,0.08
6,source_url,1,0.04


In [33]:
df.duplicated().sum()

np.int64(0)

## 3. Fix Data Types

In [34]:
# pct_workforce is currently a string with percentage signs. We need to clean it and convert it to a float for analysis.

df["pct_workforce"] = (
    df["pct_workforce"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.strip()
    .replace("nan", np.nan)
    .astype(float)
)
print("pct_workforce → float ✅")
df["pct_workforce"].describe()

pct_workforce → float ✅


count    1530.000000
mean       30.047059
std        30.535665
min         0.000000
25%        10.000000
50%        19.000000
75%        33.000000
max       100.000000
Name: pct_workforce, dtype: float64

In [36]:
# raised_mm is currently a string with dollar signs and commas. We need to clean it and convert it to a float for analysis.

df["raised_mm"] = (
    df["raised_mm"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
    .replace("nan", np.nan)
    .astype(float)
)
print("raised_mm → float ✅")
df["raised_mm"].describe()

raised_mm → float ✅


count      2178.000000
mean        964.815886
std        5692.549542
min           0.000000
25%          50.250000
50%         171.000000
75%         482.750000
max      121900.000000
Name: raised_mm, dtype: float64

In [30]:
# date is currently a string. We need to convert it to a datetime for analysis.

df["date"] = pd.to_datetime(df["date"], errors="coerce")
print("date → datetime ✅")
df["date"].dtype

date → datetime ✅


dtype('<M8[us]')

## 4. Handle Missing Values

**Strategy:**
- `layoff_count`: Flag missing as `layoff_count_missing`, then impute with industry median
- `pct_workforce`: Flag missing as `pct_workforce_missing`, then impute with industry median
- `raised_mm`: Flag missing as `raised_mm_missing`, then impute with stage median

In [37]:
# layoff_count is currently a float with some nulls. We will impute the nulls with the median layoff count for that industry. If the industry is also missing, we will use the overall median layoff count.

df["layoff_count_missing"] = df["layoff_count"].isnull().astype(bool)
print("Missing flag created ✅")

industry_median_layoffs = df.groupby("industry")["layoff_count"].median()
df["layoff_count"] = df.apply(
    lambda row: industry_median_layoffs.get(row["industry"], df["layoff_count"].median())
    if pd.isnull(row["layoff_count"]) else row["layoff_count"],
    axis=1
)
print(f"layoff_count nulls remaining: {df['layoff_count'].isnull().sum()} ✅")

Missing flag created ✅
layoff_count nulls remaining: 0 ✅


In [38]:
# pct_workforce is currently a float with some nulls. We will impute the nulls with the median percentage for that industry. If the industry is also missing, we will use the overall median percentage.

df["pct_workforce_missing"]    = df["pct_workforce"].isnull().astype(bool)
print("Missing flag created ✅")

industry_median_pct = df.groupby("industry")["pct_workforce"].median()
df["pct_workforce"] = df.apply(
    lambda row: industry_median_pct.get(row["industry"], df["pct_workforce"].median())
    if pd.isnull(row["pct_workforce"]) else row["pct_workforce"],
    axis=1
)
print(f"pct_workforce nulls remaining: {df['pct_workforce'].isnull().sum()} ✅")

Missing flag created ✅
pct_workforce nulls remaining: 0 ✅


In [39]:
# raised_mm is currently a float with some nulls. We will impute the nulls with the median raised amount for that stage. If the stage is also missing, we will use the overall median raised amount.

df["raised_mm_missing"]    = df["raised_mm"].isnull().astype(bool)
print("Missing flag created ✅")

stage_median_raised = df.groupby("stage")["raised_mm"].median()
df["raised_mm"] = df.apply(
    lambda row: stage_median_raised.get(row["stage"], df["raised_mm"].median())
    if pd.isnull(row["raised_mm"]) else row["raised_mm"],
    axis=1
)
print(f"raised_mm nulls remaining: {df['raised_mm'].isnull().sum()} ✅")


Missing flag created ✅
raised_mm nulls remaining: 0 ✅


## 5. Standardize Text Fields

In [41]:
# Standardize text columns by stripping whitespace and converting to title case

text_cols = ["company", "location", "industry", "stage", "country"]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

# Fill industry nulls with 'Other' since it's a categorical column and we don't want to drop rows.
print(df['industry'].unique())

df['industry'] = df['industry'].fillna('Other')

print('Industry nulls filled with "Other" ✅')

<StringArray>
[     'Consumer',  'Transportat…',       'Support',        'Travel',
     'Marketing',        'Retail',       'Fitness',      'Security',
    'Recruiting',         'Media', 'Infrastructu…',   'Real Estate',
          'Data',       'Finance',          'Food',            'Hr',
       'Product',     'Education',        'Energy',         'Other',
     'Aerospace',  'Construction',         'Legal',     'Logistics',
        'Crypto',    'Healthcare',         'Sales',      'Hardware',
   'Manufactur…',            'Ai']
Length: 30, dtype: str
Industry nulls filled with "Other" ✅


In [ ]:
# Fill country nulls based on location clues, then clean up the location text to remove "Non-U.S." suffixes.
df.loc[(df['country'].isnull()) & (df['location'] == "Montreal Non-U.S."), 'country'] = 'Canada'
df.loc[(df['country'].isnull()) & (df['location'] == "Berlin Non-U.S."), 'country'] = 'Germany'

print('Country nulls filled based on location clues ✅')

In [47]:
# Fixing wrong country for Sf Bay Area entries

print(df.loc[df['location'] == 'Sf Bay Area', 'country'].unique())
print()
df.loc[df['location'] == 'Sf Bay Area', 'country'] = 'US'
print('Sf Bay Area country fixed to US ✅')

# United arab emirates and United kingdom have some variations in the country column that we need to clean up for consistency.
df['country'] = df['country'].replace({'United Arab E…': 'UAE', 'Uae': 'UAE'})
df['country'] = df['country'].replace({'United States': 'US', 'United Kingdo…': 'UK'})
print('Country names standardized ✅')

<StringArray>
['US']
Length: 1, dtype: str
Sf Bay Area country fixed to US ✅
Country names standardized ✅


In [ ]:
# Cleaning up the location text to remove "Non-U.S." suffixes.
df['location'] = df['location'].str.replace(r'\s*Non-U\.S\.\s*', '', regex=True, case=False)

# Dropping  null values for stage column
df = df.dropna(subset=['stage'])

# Dropping source url column
df = df.drop(columns=['source_url'])

print()
print("Text columns standardized ✅")
df[text_cols].head()
print()

## 6. Feature Engineering

In [31]:
df["year"]    = df["date"].dt.year
df["month"]   = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["year_quarter"] = df["date"].dt.to_period("Q").astype(str)

print("Date features created ✅")
df[["date", "year", "month", "quarter", "year_quarter"]].head()

Date features created ✅


,date,year,month,quarter,year_quarter
0,2020-03-13,2020,3,1,2020Q1
1,2020-03-13,2020,3,1,2020Q1
2,2020-03-16,2020,3,1,2020Q1
3,2020-03-16,2020,3,1,2020Q1
4,2020-03-16,2020,3,1,2020Q1


## 7. Final Validation & Export

In [48]:
df = df.drop(columns =['layoff_count_missing' , 'raised_mm_missing', 'pct_workforce_missing'])
print("Missing flags dropped ✅")

Missing flags dropped ✅


In [49]:
print("=== Final Shape ===")
print(df.shape)

print("\n=== Remaining Nulls ===")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n=== Data Types ===")
print(df.dtypes)

df.head()

=== Final Shape ===
(2465, 14)

=== Remaining Nulls ===
Series([], dtype: int64)

=== Data Types ===
company                     str
location                    str
layoff_count            float64
date             datetime64[us]
pct_workforce           float64
industry                    str
stage                       str
raised_mm               float64
country                     str
is_ai_company              bool
year                      int32
month                     int32
quarter                   int32
year_quarter                str
dtype: object


,company,location,layoff_count,date,pct_workforce,industry,stage,raised_mm,country,is_ai_company,year,month,quarter,year_quarter
0,Panda Squad,Sf Bay Area,6.0,2020-03-13,75.0,Consumer,Seed,1.0,US,False,2020,3,1,2020Q1
1,Hopskipdrive,Los Angeles,8.0,2020-03-13,10.0,Transportat…,Unknown,45.0,US,False,2020,3,1,2020Q1
2,Help.Com,Austin,16.0,2020-03-16,100.0,Support,Seed,6.0,US,False,2020,3,1,2020Q1
3,Service,Los Angeles,109.0,2020-03-16,100.0,Travel,Seed,5.0,US,False,2020,3,1,2020Q1
4,Inspirato,Denver,130.0,2020-03-16,22.0,Travel,Series C,79.0,US,False,2020,3,1,2020Q1


In [18]:
df.to_csv("../data/cleaned_layoff_dataset.csv", index=False)
print("Cleaned dataset exported to ../data/cleaned_layoff_dataset.csv ✅")

Cleaned dataset exported to ../data/cleaned_layoff_dataset.csv ✅
